# 01 · Data Exploration

Pull all three data layers for a single ticker and visualise the raw inputs  
that feed the valuation models — financials, market, and macro.  
Run this first to sanity-check data quality before running any model.

**No API keys required** — yfinance + CBOE public data cover everything here.  
Set `FRED_API_KEY` in `.env` for live macro series; otherwise hard-coded defaults are shown.

In [1]:
import sys
import pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parent))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from fairprice.data import FinancialsClient, MarketClient, MacroClient

TICKER = 'AAPL'

fin = FinancialsClient()
mkt = MarketClient()
mac = MacroClient()

print(f'Fetching data for {TICKER}...')

Fetching data for AAPL...


In [2]:
profile  = fin.get_profile(TICKER)
stmts    = fin.get_statements(TICKER)
market   = mkt.get_market_data(TICKER)
macro    = mac.get_macro_data()

print(f'{profile.name}  |  {profile.sector}  /  {profile.industry}')
print(f'Exchange: {profile.exchange}  |  Currency: {profile.currency}')
print(f'Data source: {stmts.source}')
print(f'Income stmt rows: {len(stmts.income_statement)}')
print(f'Cash flow rows  : {len(stmts.cash_flow)}')
print(f'Balance sheet   : {len(stmts.balance_sheet)}')

Apple Inc.  |  Technology  /  Consumer Electronics
Exchange: NMS  |  Currency: USD
Data source: yfinance
Income stmt rows: 5
Cash flow rows  : 5
Balance sheet   : 5


## 1 · Financial Performance

Revenue, net income, and free cash flow over the last five years.  
**FCF conversion** (FCF / Net Income) shows how much of reported profit  
actually becomes cash — values above 1.0 are a quality signal.

In [3]:
inc = stmts.income_statement.tail(5)
cf  = stmts.cash_flow.tail(5)

years = [d.strftime('%Y') for d in inc.index]

def _s(df, col, scale=1e9):
    return (df[col] / scale).tolist() if col in df.columns else [None] * len(df)

revenue  = _s(inc, 'Total Revenue')
net_inc  = _s(inc, 'Net Income')
fcf      = _s(cf,  'Free Cash Flow')

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Revenue / Net Income / FCF  ($B)', 'Profit Margins (%)'),
    horizontal_spacing=0.12,
)

colors = {'Revenue': '#2196F3', 'Net Income': '#4CAF50', 'FCF': '#FF9800'}
for name, vals in [('Revenue', revenue), ('Net Income', net_inc), ('FCF', fcf)]:
    fig.add_trace(go.Bar(name=name, x=years, y=vals,
                         marker_color=colors[name]), row=1, col=1)

# Margins
gross_m = [g/r*100 if r else None for g, r in zip(_s(inc, 'Gross Profit'), revenue)]
op_m    = [o/r*100 if r else None for o, r in zip(_s(inc, 'Operating Income'), revenue)]
net_m   = [n/r*100 if r else None for n, r in zip(net_inc, revenue)]

for name, vals, color in [('Gross', gross_m, '#2196F3'),
                           ('Operating', op_m, '#4CAF50'),
                           ('Net', net_m, '#FF9800')]:
    fig.add_trace(go.Scatter(name=f'{name} margin', x=years, y=vals,
                             mode='lines+markers', marker_color=color), row=1, col=2)

fig.update_layout(title_text=f'{profile.name} — Income Statement', height=420,
                  barmode='group', legend=dict(orientation='h', y=-0.15))
fig.update_yaxes(title_text='$B', row=1, col=1)
fig.update_yaxes(title_text='%', row=1, col=2)
fig.show()

In [4]:
# FCF quality: conversion ratio and capex intensity
op_cf  = _s(cf, 'Operating Cash Flow')
capex  = [abs(v) if v is not None else None for v in _s(cf, 'Capital Expenditure')]
fcf_conv = [f/n if (f and n and n > 0) else None for f, n in zip(fcf, net_inc)]

fig = make_subplots(rows=1, cols=2,
    subplot_titles=('Operating CF vs CapEx ($B)', 'FCF Conversion (FCF / Net Income)'),
    horizontal_spacing=0.12)

fig.add_trace(go.Bar(name='Operating CF', x=years, y=op_cf, marker_color='#4CAF50'), row=1, col=1)
fig.add_trace(go.Bar(name='CapEx',        x=years, y=capex,  marker_color='#F44336'), row=1, col=1)
fig.add_trace(go.Bar(name='FCF Conv.',    x=years, y=fcf_conv, marker_color='#9C27B0',
                     text=[f'{v:.2f}x' if v else '' for v in fcf_conv],
                     textposition='outside'), row=1, col=2)

fig.add_hline(y=1.0, line_dash='dash', line_color='gray',
              annotation_text='1.0× (FCF = Net Income)', row=1, col=2)

fig.update_layout(title_text='Cash Flow Quality', height=380, barmode='group',
                  legend=dict(orientation='h', y=-0.15))
fig.show()

avg_conv = pd.Series([v for v in fcf_conv if v]).mean()
quality  = 'excellent' if avg_conv > 1.1 else 'good' if avg_conv > 0.9 else 'below average'
print(f'Average FCF conversion: {avg_conv:.2f}× — {quality}')

Average FCF conversion: 1.05× — good


## 2 · Balance Sheet Strength

Debt load and cash position determine the spread between enterprise value  
and equity value in the DCF. High net debt reduces intrinsic per-share value.

In [5]:
bs = stmts.balance_sheet.tail(5)
bs_years = [d.strftime('%Y') for d in bs.index]

def _bs(col): return _s(bs, col)

cash  = _bs('Cash And Cash Equivalents') or _bs('Cash Cash Equivalents And Short Term Investments')
debt  = _bs('Total Debt') or _bs('Long Term Debt')
equity = _bs('Total Stockholders Equity') or _bs('Stockholders Equity')

net_debt = [d - c if (d is not None and c is not None) else None
            for d, c in zip(debt or [], cash or [])]

fig = make_subplots(rows=1, cols=2,
    subplot_titles=('Debt vs Cash ($B)', 'Net Debt Trend ($B)'),
    horizontal_spacing=0.12)

if debt:
    fig.add_trace(go.Bar(name='Total Debt', x=bs_years, y=debt, marker_color='#F44336'), row=1, col=1)
if cash:
    fig.add_trace(go.Bar(name='Cash',       x=bs_years, y=cash, marker_color='#4CAF50'), row=1, col=1)

if net_debt:
    colors_nd = ['#F44336' if v and v > 0 else '#4CAF50' for v in net_debt]
    fig.add_trace(go.Bar(name='Net Debt', x=bs_years, y=net_debt,
                         marker_color=colors_nd,
                         text=[f'${v:.1f}B' if v else '' for v in net_debt],
                         textposition='outside'), row=1, col=2)
    fig.add_hline(y=0, line_dash='dash', line_color='gray', row=1, col=2)

fig.update_layout(title_text='Balance Sheet', height=380, barmode='group',
                  legend=dict(orientation='h', y=-0.15))
fig.show()

if net_debt and net_debt[-1] is not None:
    label = 'net cash (positive for equity holders)' if net_debt[-1] < 0 else 'net debt'
    print(f'Latest net debt: ${net_debt[-1]:.1f}B — {label}')

Latest net debt: $62.7B — net debt


## 3 · Price History & Volatility

In [6]:
prices = market.prices.copy()
prices['MA50']  = prices['Close'].rolling(50).mean()
prices['MA200'] = prices['Close'].rolling(200).mean()

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    row_heights=[0.72, 0.28],
                    subplot_titles=(f'{TICKER} Price (5Y)', 'Volume'))

fig.add_trace(go.Candlestick(
    x=prices.index, open=prices['Open'], high=prices['High'],
    low=prices['Low'],  close=prices['Close'],
    name='OHLC', increasing_line_color='#4CAF50',
    decreasing_line_color='#F44336'), row=1, col=1)

fig.add_trace(go.Scatter(x=prices.index, y=prices['MA50'],
    name='MA50', line=dict(color='#FF9800', width=1.5)), row=1, col=1)
fig.add_trace(go.Scatter(x=prices.index, y=prices['MA200'],
    name='MA200', line=dict(color='#2196F3', width=1.5, dash='dash')), row=1, col=1)

vol_color = ['#4CAF50' if c >= o else '#F44336'
             for c, o in zip(prices['Close'], prices['Open'])]
fig.add_trace(go.Bar(x=prices.index, y=prices['Volume'],
    name='Volume', marker_color=vol_color, showlegend=False), row=2, col=1)

fig.update_layout(height=550, xaxis_rangeslider_visible=False,
                  legend=dict(orientation='h', y=1.05))
fig.show()

daily_ret = prices['Close'].pct_change().dropna()
ann_vol   = daily_ret.std() * (252 ** 0.5)
total_ret = (prices['Close'].iloc[-1] / prices['Close'].iloc[0] - 1) * 100
print(f'5Y total return  : {total_ret:+.1f}%')
print(f'Annualised vol   : {ann_vol*100:.1f}%')
print(f'Beta (1Y vs SPY) : {market.beta_1y:.2f}')
print(f'Current VIX      : {market.vix:.1f}')

5Y total return  : +150.3%
Annualised vol   : 27.4%
Beta (1Y vs SPY) : 0.97
Current VIX      : 16.6


## 4 · Macro Context

Risk-free rate and equity risk premium are the two most powerful levers  
in a DCF — a 1pp rise in the 10Y yield can move intrinsic value by 15–25%.  
This section shows where we stand relative to history.

In [7]:
macro_vals = {
    'Risk-Free Rate\n(10Y Treasury)': macro.risk_free_rate * 100,
    'Equity Risk\nPremium': macro.equity_risk_premium * 100,
    'Inflation\n(CPI YoY)': macro.inflation_rate * 100,
    'Real GDP\nGrowth': macro.real_gdp_growth * 100,
    'Fed Funds\nRate': macro.fed_funds_rate * 100,
    'Unemployment': macro.unemployment_rate * 100,
}

fig = go.Figure(go.Bar(
    x=list(macro_vals.keys()),
    y=list(macro_vals.values()),
    marker_color=['#2196F3', '#9C27B0', '#FF9800', '#4CAF50', '#F44336', '#00BCD4'],
    text=[f'{v:.2f}%' for v in macro_vals.values()],
    textposition='outside',
))
fig.update_layout(title='Macro Inputs to Valuation Models',
                  yaxis_title='%', height=380)
fig.show()

In [8]:
# Yield curve (requires FRED key; falls back to defaults)
curve = mac.get_yield_curve()

labels = {'3m': '3M', '2y': '2Y', '5y': '5Y', '10y': '10Y', '30y': '30Y'}
maturities = [labels[k] for k in curve]
yields     = [v * 100 for v in curve.values()]

is_inverted = curve.get('2y', 0) > curve.get('10y', 0)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=maturities, y=yields,
    mode='lines+markers+text',
    text=[f'{y:.2f}%' for y in yields],
    textposition='top center',
    line=dict(color='#F44336' if is_inverted else '#2196F3', width=2),
    fill='tozeroy', fillcolor='rgba(33,150,243,0.1)'
))
fig.update_layout(
    title=f'US Treasury Yield Curve  '
          f'({"⚠ INVERTED — historically recession-predictive" if is_inverted else "Normal slope"})',
    yaxis_title='Yield (%)', height=360
)
fig.show()

spread_10_2 = (curve.get('10y', 0) - curve.get('2y', 0)) * 100
print(f'10Y-2Y spread: {spread_10_2:+.0f} bps  '
      f'({"inverted — watch for recessionary pressure" if spread_10_2 < 0 else "positive — normal"})')

10Y-2Y spread: +0 bps  (positive — normal)


## 5 · Data Quality Summary

Before running any model, check completeness — missing line items  
force the models to fall back to proxies and reduce confidence.

In [9]:
checks = {
    'Income statement present'  : not stmts.income_statement.empty,
    'Cash flow present'         : not stmts.cash_flow.empty,
    'Balance sheet present'     : not stmts.balance_sheet.empty,
    'TTM metrics available'     : len(stmts.ttm) >= 3,
    'FCF positive (TTM)'        : stmts.ttm.get('Free Cash Flow', 0) > 0,
    'Revenue > 5 years'         : len(stmts.income_statement) >= 5,
    'Price history loaded'      : not market.prices.empty,
    'Beta computed'             : 0.1 < market.beta_1y < 4.0,
    'FRED macro loaded'         : macro.risk_free_rate > 0,
}

score = sum(checks.values()) / len(checks)

df_checks = pd.DataFrame({'Check': list(checks.keys()),
                           'Pass': ['✓' if v else '✗' for v in checks.values()]})
print(df_checks.to_string(index=False))
print(f'\nData completeness score: {score:.0%}')
if score < 0.75:
    print('⚠  Low completeness — confidence scores will be penalised.')
else:
    print('✓  Good data quality — all three valuation models should run.')

                   Check Pass
Income statement present    ✓
       Cash flow present    ✓
   Balance sheet present    ✓
   TTM metrics available    ✓
      FCF positive (TTM)    ✓
       Revenue > 5 years    ✓
    Price history loaded    ✓
           Beta computed    ✓
       FRED macro loaded    ✓

Data completeness score: 100%
✓  Good data quality — all three valuation models should run.
